# RF-DETR 학습 + 평가 (resolution=616, aug Top-3)

**환경**: Colab A100  
**모델**: RF-DETR Base (rfdetr 패키지)  
**조건**: epochs=15, batch=16, resolution=616 (56 배수 강제), seed=42  
**증강**: CLAHE / GaussianBlur / MotionBlur (ablation Top-3)  

## 노트북 구조
- **학습 파트** (셀 1~5): 데이터 준비 → 모델 → 학습 → 백업
- **평가 파트** (셀 6~8): 환경 셋업 → 33장 평가 → JSON 저장

평가 파트는 자체 완결되어 있어 런타임 끊겨도 단독 실행 가능.


## 셀 1 — 환경 + 패키지 설치 + Drive 마운트


In [ ]:
# rfdetr 패키지 (코어 + train extras)
!pip install -q rfdetr
!pip install -q "rfdetr[train,loggers]"

import torch, gc, json, zipfile, shutil, os
import numpy as np
from pathlib import Path
from PIL import Image

# Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# GPU 확인
!nvidia-smi | head -20


## 셀 2 — 학습 데이터 재배치 (RF-DETR 구조) + CLASS_NAMES 로드

YOLO 구조 → RF-DETR 구조 (`train/images`, `valid/images`)로 심볼릭 링크 + `data.yaml` 생성


In [ ]:
import time, zipfile, shutil, os, yaml
from pathlib import Path

ZIP = '/content/drive/MyDrive/dataset_yolo.zip'
EXTRACT_TO = Path('/content')
DATA_FLAT = Path('/content')                     # zip 풀린 후 위치
RFDETR_DATA = Path('/content/rfdetr_data')

# 1) 압축 풀기 (이미 있으면 스킵)
if (DATA_FLAT / 'images' / 'train').exists() and len(list((DATA_FLAT / 'images' / 'train').iterdir())) > 0:
    print('스킵: 데이터 이미 풀려있음')
else:
    t0 = time.time()
    print('압축 풀기 시작...')
    with zipfile.ZipFile(ZIP, 'r') as z:
        z.extractall(EXTRACT_TO)
    print(f'완료: {(time.time()-t0)/60:.1f}분')

# 2) 검증
for split in ['train', 'val']:
    n_img = len(list((DATA_FLAT / 'images' / split).glob('*.*')))
    n_lbl = len(list((DATA_FLAT / 'labels' / split).glob('*.txt')))
    print(f'  {split}: 이미지 {n_img:,} / 라벨 {n_lbl:,}')

# 3) RF-DETR 구조로 심볼릭 링크 (디스크 절약)
if RFDETR_DATA.exists():
    shutil.rmtree(RFDETR_DATA)
RFDETR_DATA.mkdir()

split_map = {'train': 'train', 'val': 'valid'}   # RF-DETR은 'valid' 이름 요구
for our_split, rf_split in split_map.items():
    rf_split_dir = RFDETR_DATA / rf_split
    rf_split_dir.mkdir()
    # 심링크: rfdetr_data/train/images → /content/images/train
    os.symlink(DATA_FLAT / 'images' / our_split, rf_split_dir / 'images')
    os.symlink(DATA_FLAT / 'labels' / our_split, rf_split_dir / 'labels')

# 4) data.yaml 생성 (RF-DETR 요구 이름)
with open(DATA_FLAT / 'pill.yaml', 'r', encoding='utf-8') as f:
    pill_yaml = yaml.safe_load(f)
names = pill_yaml['names']
if isinstance(names, dict):
    names = [names[i] for i in sorted(names.keys())]

CLASS_NAMES = list(names)
NUM_CLASSES = len(CLASS_NAMES)

data_yaml = {
    'path': str(RFDETR_DATA),
    'train': 'train/images',
    'val':   'valid/images',
    'nc':    NUM_CLASSES,
    'names': {i: n for i, n in enumerate(CLASS_NAMES)},
}
with open(RFDETR_DATA / 'data.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(data_yaml, f, allow_unicode=True, sort_keys=False)

# 5) 검증
from rfdetr.datasets.yolo import is_valid_yolo_dataset
valid = is_valid_yolo_dataset(str(RFDETR_DATA))
print(f'\nRF-DETR 구조 검증: {valid}')

# 구조 출력
print(f'\n구조:')
print(f'  {RFDETR_DATA}/')
for p in sorted(RFDETR_DATA.iterdir()):
    if p.is_dir():
        print(f'    {p.name}/')
        for sub in sorted(p.iterdir()):
            n = len(list(sub.iterdir())) if sub.is_dir() else '-'
            print(f'      {sub.name}/  ({n}개)')
    else:
        print(f'    {p.name}')
print(f'  CLASS_NAMES: {len(CLASS_NAMES)}개')

## 셀 3 — 사전학습 모델 인스턴스 (num_classes=118)


In [ ]:
# 사전학습 모델 인스턴스 (num_classes=118)
from rfdetr import RFDETRBase

model = RFDETRBase(num_classes=118)
print(f'✅ 모델 생성')
print(f'  num_classes: {model.model_config.num_classes}')
print(f'  resolution:  {model.model_config.resolution}')
print(f'  patch_size:  {model.model_config.patch_size}')
print(f'  num_windows: {model.model_config.num_windows}')


## 셀 4 — 학습 (resolution=616, aug Top-3, 15 epochs)

A100 기준 약 132분 소요. resolution은 RF-DETR 제약(56 배수)으로 616.


In [ ]:
import torch, time

aug_config = {
    "CLAHE": {"clip_limit": 4.0, "tile_grid_size": (8, 8), "p": 0.5},
    "GaussianBlur": {"blur_limit": 7, "p": 0.3},
    "MotionBlur": {"blur_limit": 7, "p": 0.3},
}

EXP_NAME = 'exp_rfdetr'
OUTPUT_DIR = f'/content/runs/{EXP_NAME}'

t0 = time.time()
model.train(
    dataset_dir='/content/rfdetr_data',
    dataset_file='yolo',
    epochs=15,
    batch_size=16,
    grad_accum_steps=1,
    resolution=616,                 # 56 배수
    output_dir=OUTPUT_DIR,
    seed=42,
    aug_config=aug_config,
    class_names=CLASS_NAMES,
    num_workers=4,
    tensorboard=False,
    wandb=False,
)

elapsed = (time.time() - t0) / 60
print(f'\n학습 완료: {elapsed:.1f}분')
print(f'output_dir: {OUTPUT_DIR}')

## 셀 5 — Drive 백업 (학습 산출물)


In [ ]:
import shutil, json, os
from pathlib import Path

EXP_NAME_EVAL = 'exp_rfdetr'
AUTHOR = 'wony98'

LOCAL_RUN = Path('/content/runs') / EXP_NAME_EVAL
DRIVE_BACKUP = Path('/content/drive/MyDrive/pill_detection/results') / EXP_NAME_EVAL

# Drive 백업
DRIVE_BACKUP.parent.mkdir(parents=True, exist_ok=True)
if DRIVE_BACKUP.exists():
    print(f'⚠️ 기존 백업 존재: {DRIVE_BACKUP} (덮어쓰기)')
    shutil.rmtree(DRIVE_BACKUP)
shutil.copytree(LOCAL_RUN, DRIVE_BACKUP)
print(f'✅ Drive 백업 완료: {DRIVE_BACKUP}\n')

# 폴더 안 파일 목록
print('산출물:')
for f in sorted(DRIVE_BACKUP.rglob('*')):
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        print(f'  {f.relative_to(DRIVE_BACKUP)}  ({size_mb:.1f} MB)')

# best 가중치 자동 탐지
candidates = list(LOCAL_RUN.rglob('*.pth')) + list(LOCAL_RUN.rglob('*.pt'))
best_candidates = [p for p in candidates if 'best' in p.name.lower()]
if best_candidates:
    BEST_PT = best_candidates[0]
    # ema 우선
    for p in best_candidates:
        if 'ema' in p.name.lower():
            BEST_PT = p
            break
else:
    BEST_PT = candidates[0] if candidates else None

print(f'\n사용할 best 가중치: {BEST_PT}')

## 셀 6 — 평가 환경 셋업 (자체 완결, 런타임 끊겼을 경우 여기부터 시작)

best EMA 가중치(`checkpoint_best_ema.pth`)로 평가 모델 로드.  
학습 직후 이어서 돌려도 OK, 런타임 끊겨서 단독 실행해도 OK.


In [ ]:
# === 1) 패키지 설치 ===
!pip install -q rfdetr
!pip install -q "rfdetr[train,loggers]"

# === 2) imports ===
import torch, gc, json, zipfile, shutil, os
import numpy as np
from pathlib import Path
from PIL import Image
from rfdetr import RFDETRBase

# === 3) Drive 마운트 ===
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

# === 4) eval_package 풀기 (한글 파일명 정상) ===
EVAL_ZIP = '/content/drive/MyDrive/eval_package.zip'
EVAL_ROOT = Path('/content/eval_package')
if EVAL_ROOT.exists():
    shutil.rmtree(EVAL_ROOT)
with zipfile.ZipFile(EVAL_ZIP, 'r') as z:
    z.extractall(EVAL_ROOT)

# answer_key
with open(EVAL_ROOT / 'answer_key.json', 'r', encoding='utf-8') as f:
    ANSWER_KEY = json.load(f)
test_imgs = sorted(EVAL_ROOT.rglob('*.png'))
matched = {p.name for p in test_imgs} & set(ANSWER_KEY.keys())
print(f'평가 이미지: {len(test_imgs)}장 / 매칭: {len(matched)}/33')

# === 5) CLASS_NAMES (pill.yaml에서) ===
# pill.yaml은 Drive 백업 폴더에서 가져오기
import yaml
PILL_YAML_CANDIDATES = [
    '/content/pill.yaml',
    '/content/drive/MyDrive/pill.yaml',
]
pill_yaml_path = None
for p in PILL_YAML_CANDIDATES:
    if os.path.exists(p):
        pill_yaml_path = p
        break

if pill_yaml_path is None:
    # dataset_yolo.zip 에서 추출
    print('pill.yaml 없음 → dataset_yolo.zip에서 추출')
    with zipfile.ZipFile('/content/drive/MyDrive/dataset_yolo.zip', 'r') as z:
        yaml_names = [n for n in z.namelist() if n.endswith('pill.yaml')]
        if yaml_names:
            z.extract(yaml_names[0], '/content/')
            extracted = Path('/content') / yaml_names[0]
            pill_yaml_path = '/content/pill.yaml'
            if extracted != Path(pill_yaml_path):
                shutil.copy2(extracted, pill_yaml_path)

with open(pill_yaml_path, 'r', encoding='utf-8') as f:
    pill_yaml = yaml.safe_load(f)
names = pill_yaml['names']
if isinstance(names, dict):
    names = [names[i] for i in sorted(names.keys())]
CLASS_NAMES = list(names)
NUM_CLASSES = len(CLASS_NAMES)
print(f'클래스 수: {NUM_CLASSES}')

# === 6) best.pth 로드 ===
BEST_PT = '/content/drive/MyDrive/pill_detection/results/exp_rfdetr/checkpoint_best_ema.pth'
print(f'\nbest.pth: {BEST_PT}')
print(f'exists: {os.path.exists(BEST_PT)}')

eval_model = RFDETRBase(
    num_classes=118,
    pretrain_weights=BEST_PT,
    resolution=616,
)
print(f'\n✅ 평가 모델 로드 완료 (resolution={eval_model.model_config.resolution})')

# === 7) 헬퍼: IoU 그룹핑 + Top-3 ===
def compute_iou(a, b):
    xA = max(a[0], b[0]); yA = max(a[1], b[1])
    xB = min(a[2], b[2]); yB = min(a[3], b[3])
    inter = max(0.0, xB - xA) * max(0.0, yB - yA)
    area_a = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    area_b = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])
    union = area_a + area_b - inter
    return inter / union if union > 1e-9 else 0.0


def group_and_top3(boxes, class_ids, scores, class_names, iou_thr=0.5, max_top=3):
    if len(boxes) == 0:
        return []
    order = np.argsort(-scores)
    bs = boxes[order].tolist()
    ls = class_ids[order].tolist()
    ss = scores[order].tolist()

    used = [False] * len(bs)
    detections = []
    for i in range(len(bs)):
        if used[i]:
            continue
        group_idx = [i]
        used[i] = True
        for j in range(i + 1, len(bs)):
            if used[j]:
                continue
            if compute_iou(bs[i], bs[j]) > iou_thr:
                group_idx.append(j)
                used[j] = True

        cls_best = {}
        for k in group_idx:
            cls = int(ls[k])
            if (cls not in cls_best) or (ss[k] > cls_best[cls][0]):
                cls_best[cls] = (ss[k], k)
        top = sorted(cls_best.items(), key=lambda x: -x[1][0])[:max_top]
        rep_box = bs[top[0][1][1]]

        names = []
        idxs = []
        confs = []
        for c, (s, _) in top:
            if 0 <= c < len(class_names):
                names.append(class_names[c])
                idxs.append(int(c))
                confs.append(float(s))
        if names:
            detections.append({
                'bbox': [float(x) for x in rep_box],
                'top3_class_idx':  idxs,
                'top3_class_name': names,
                'top3_confidence': confs,
            })
    return detections

print('\n✅ 헬퍼 함수 정의 완료')
print('\n→ 다음: 33장 평가 셀 실행')

## 셀 7 — 33장 평가 (Top-1 / Top-3)

IoU 그룹핑 + Top-3 추출. 이름 기반 채점 (class_idx 매핑 무관).


In [ ]:
import time, json
import numpy as np
from PIL import Image
from datetime import datetime

results = []
top1_hit = 0
top3_hit = 0
detected_n = 0
n = len(test_imgs)

t0 = time.time()
for img_path in test_imgs:
    fname = img_path.name
    if fname not in ANSWER_KEY:
        continue

    gt = ANSWER_KEY[fname]
    gt_dl_name = gt['dl_name']
    gt_class_idx = gt['class_idx']

    img_rgb = np.array(Image.open(img_path).convert('RGB'))
    H, W = img_rgb.shape[:2]
    sv_dets = eval_model.predict(img_rgb, threshold=0.01)

    detections = group_and_top3(
        sv_dets.xyxy, sv_dets.class_id, sv_dets.confidence, CLASS_NAMES
    )

    if len(detections) > 0:
        detected_n += 1
        top1_names = [d['top3_class_name'][0] for d in detections if d.get('top3_class_name')]
        top3_names = [nm for d in detections for nm in d.get('top3_class_name', [])]

        if gt_dl_name in top1_names:
            top1_hit += 1
        if gt_dl_name in top3_names:
            top3_hit += 1

    results.append({
        'filename': fname,
        'img_width': W, 'img_height': H,
        'gt_dl_name': gt_dl_name,
        'gt_class_idx': gt_class_idx,
        'num_detections': len(detections),
        'detections': detections,
    })

elapsed = time.time() - t0
print(f'평가 완료: {elapsed:.1f}초 ({elapsed/n:.2f}s/img)')
print(f'Detection rate: {detected_n}/{n} = {detected_n/n:.3f}')
print(f'Top-1 accuracy: {top1_hit}/{n}    = {top1_hit/n:.3f}')
print(f'Top-3 accuracy: {top3_hit}/{n}    = {top3_hit/n:.3f}')

## 셀 8 — JSON 저장 + Drive 백업

`_model_collection/` 폴더에 백업 → 로컬 노트북(compare_experiments)에서 다른 모델과 비교


In [ ]:
import json, shutil
from datetime import datetime
from pathlib import Path

# 학습 시 출력된 파라미터 수 그대로 사용 (LWDETR 32.3M)
total_params_M = 32.3

output_data = {
    'exp_name': 'exp_rfdetr',
    'author': 'wony98',
    'experiment_type': 'model',
    'model_framework': 'rfdetr',
    'model_name': 'RFDETRBase',
    'model_params_M': total_params_M,
    'batch_size_used': 16,
    'imgsz': 616,
    'epochs': 15,
    'seed': 42,
    'timestamp': datetime.now().isoformat(timespec='seconds'),
    'num_images': n,
    'detection_rate': round(detected_n / n, 4),
    'top1_accuracy':  round(top1_hit / n, 4),
    'top3_accuracy':  round(top3_hit / n, 4),
    'note': 'resolution=616 (RF-DETR requires divisible by 56; 640 not allowed)',
    'results': results,
}

# 1) 로컬 저장
EVAL_OUT = Path('/content/runs/exp_rfdetr/eval')
EVAL_OUT.mkdir(parents=True, exist_ok=True)
JSON_PATH = EVAL_OUT / 'exp_rfdetr_results.json'
with open(JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)
print(f'✅ 로컬 저장: {JSON_PATH}')

# 2) Drive 모델 비교용 폴더에 백업
DRIVE_COLLECTION = Path('/content/drive/MyDrive/pill_detection/results/_model_collection')
DRIVE_COLLECTION.mkdir(parents=True, exist_ok=True)
DRIVE_JSON = DRIVE_COLLECTION / 'exp_rfdetr_results.json'
shutil.copy2(JSON_PATH, DRIVE_JSON)
print(f'✅ Drive 백업: {DRIVE_JSON}')

# 3) 두 모델 결과 한 번에 확인
print('\n' + '=' * 70)
print('  본인 담당 두 모델 결과')
print('=' * 70)
print(f'{"모델":25s} | Detection | Top-1   | Top-3')
print('-' * 70)
fr_path = DRIVE_COLLECTION / 'exp_faster_rcnn_results.json'
if fr_path.exists():
    with open(fr_path) as f:
        fr = json.load(f)
    print(f'{"Faster R-CNN v2":25s} | {fr["detection_rate"]:.3f}     | {fr["top1_accuracy"]:.3f}   | {fr["top3_accuracy"]:.3f}')
print(f'{"RF-DETR Base":25s} | {detected_n/n:.3f}     | {top1_hit/n:.3f}   | {top3_hit/n:.3f}')
print('=' * 70)